In [1]:
from datasets import load_dataset
import matplotlib.pyplot as plt

/Users/prinks/anaconda3/envs/motiondet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("mspitzna/physicsgen",name='ball_roll',trust_remote_code=True)

Generating train split: 27840 examples [00:11, 2430.04 examples/s]
Generating test split: 1800 examples [00:00, 4626.52 examples/s]
Generating validation split: 50 examples [00:00, 2798.48 examples/s]


In [5]:
train = dataset['train']
test = dataset['test']
val = dataset['validation']

In [6]:
sample = train[0]
print(sample.keys())


dict_keys(['ImgName', 'StartHeight', 'GroundIncli', 'InputTime', 'TargetTime', 'input_image', 'target_image'])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import numpy as np

# ==== Custom Dataset ====
class BallRollDataset(Dataset):
    def __init__(self, split="train",dataset):
        # Load local Parquet or use HF loader if possible
        #dataset = load_dataset("mspitzna/physicsgen", name="ball_roll", trust_remote_code=True)[split]
        self.samples = dataset
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        sample = self.samples[idx]
        input_img = torch.tensor(sample["input_image"], dtype=torch.float32)
        target_img = torch.tensor(sample["target_image"], dtype=torch.float32)

        # If images are (H, W, C), convert to (C, H, W)
        if input_img.ndim == 3 and input_img.shape[-1] <= 4:
            input_img = input_img.permute(2, 0, 1)
            target_img = target_img.permute(2, 0, 1)

        # Normalize images if needed (use /255 if uint8 images):
        input_img = input_img / 255.0
        target_img = target_img / 255.0

        return input_img, target_img

# ==== Encoder/Decoder ====
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64*16*16, latent_dim)
        self.fc_logvar = nn.Linear(64*16*16, latent_dim)
    def forward(self, x):
        h = self.conv(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64*16*16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 64, 16, 16)
        return self.deconv(h)

# ==== Neural ODE block ====
class LatentODEFunc(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim+1, 128), nn.Tanh(),
            nn.Linear(128, latent_dim)
        )
    def forward(self, t, z):
        t = t.view(1, 1).repeat(z.size(0), 1)
        inp = torch.cat([z, t], dim=1)
        return self.net(inp)

# ==== VAE + ODE Model ====
class ODEVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.odefunc = LatentODEFunc(latent_dim)
        self.decoder = Decoder(latent_dim)
    def forward(self, x, t=[0.0, 1.0]):
        mu, logvar = self.encoder(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z0 = mu + eps * std
        t = torch.tensor(t, device=x.device)
        zT = odeint(self.odefunc, z0, t)[-1]
        x_recon = self.decoder(zT)
        return x_recon, mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    recon = F.mse_loss(recon_x, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kld

# ==== Training ====


In [ ]:
# ==== Encoder/Decoder ====
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64*16*16, latent_dim)
        self.fc_logvar = nn.Linear(64*16*16, latent_dim)
    def forward(self, x):
        h = self.conv(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64*16*16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 64, 16, 16)
        return self.deconv(h)

In [ ]:
# ==== Neural ODE block ====
class LatentODEFunc(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim+1, 128), nn.Tanh(),
            nn.Linear(128, latent_dim)
        )
    def forward(self, t, z):
        t = t.view(1, 1).repeat(z.size(0), 1)
        inp = torch.cat([z, t], dim=1)
        return self.net(inp)

# ==== VAE + ODE Model ====
class ODEVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.odefunc = LatentODEFunc(latent_dim)
        self.decoder = Decoder(latent_dim)
    def forward(self, x, t=[0.0, 1.0]):
        mu, logvar = self.encoder(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z0 = mu + eps * std
        t = torch.tensor(t, device=x.device)
        zT = odeint(self.odefunc, z0, t)[-1]
        x_recon = self.decoder(zT)
        return x_recon, mu, logvar

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    recon = F.mse_loss(recon_x, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kld


In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
latent_dim = 32
model = ODEVAE(latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

dataset = BallRollDataset("train")
loader = DataLoader(dataset, batch_size=16, shuffle=True)

num_epochs = 10
for epoch in range(num_epochs):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred, mu, logvar = model(x)
        loss = vae_loss(y_pred, y, mu, logvar)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs} Loss: {loss.item():.4f}")

print("ODE-VAE training complete.")
